In [1]:
import dgl.nn as dglnn
import dgl
from dgl import from_networkx
import torch.nn as nn
import torch as th
import torch.nn.functional as F
import dgl.function as fn
import networkx as nx
import pandas as pd
import socket
import struct
import random
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import category_encoders as ce
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [3]:
data = pd.read_csv('NF-BoT-IoT-v2-0.03.csv')
data.drop(columns=['Label'],inplace = True)
data.rename(columns={"Attack": "label"},inplace = True)
le = LabelEncoder()
le.fit_transform(data.label.values)
# data['label'] = le.transform(data['label'])
print(le.classes_)

['Benign' 'DDoS' 'DoS' 'Reconnaissance' 'Theft']


In [2]:
data = pd.read_csv('NF-BoT-IoT-v2-0.03.csv')
data['IPV4_SRC_ADDR'] = data.IPV4_SRC_ADDR.apply(lambda x: socket.inet_ntoa(struct.pack('>I', random.randint(0xac100001, 0xac1f0001))))
data['IPV4_SRC_ADDR'] = data.IPV4_SRC_ADDR.apply(str)
data['L4_SRC_PORT'] = data.L4_SRC_PORT.apply(str)
data['IPV4_DST_ADDR'] = data.IPV4_DST_ADDR.apply(str)
data['L4_DST_PORT'] = data.L4_DST_PORT.apply(str)
data['IPV4_SRC_ADDR'] = data['IPV4_SRC_ADDR'] + ':' + data['L4_SRC_PORT']
data['IPV4_DST_ADDR'] = data['IPV4_DST_ADDR'] + ':' + data['L4_DST_PORT']
data.drop(columns=['L4_SRC_PORT','L4_DST_PORT'],inplace=True)
data.drop(columns=['Label'],inplace = True)
data.rename(columns={"Attack": "label"},inplace = True)
le = LabelEncoder()
le.fit_transform(data.label.values)
data['label'] = le.transform(data['label'])
label = data.label
scaler = StandardScaler()
X_train, X_test, y_train, y_test = train_test_split(data, label, test_size=0.3, random_state=123,stratify= label)
cols_to_norm = list(set(list(X_train.iloc[:, 2:].columns ))  - set(list(['label'])) )

X_train[cols_to_norm] = scaler.fit_transform(X_train[cols_to_norm])
X_train['h'] = X_train[ cols_to_norm ].values.tolist()
G = nx.from_pandas_edgelist(X_train, "IPV4_SRC_ADDR", "IPV4_DST_ADDR", ['h','label'],create_using=nx.MultiGraph())
G = G.to_directed()
G = from_networkx(G,edge_attrs=['h','label'] )
G.ndata['h'] = th.ones(G.num_nodes(), G.edata['h'].shape[1])

X_test[cols_to_norm] = scaler.fit_transform(X_test[cols_to_norm])
X_test['h'] = X_test[ cols_to_norm ].values.tolist()
G_test = nx.from_pandas_edgelist(X_test, "IPV4_SRC_ADDR", "IPV4_DST_ADDR", ['h','label'],create_using=nx.MultiGraph())
G_test = G_test.to_directed()
G_test = from_networkx(G_test,edge_attrs=['h','label'] )
G_test.ndata['h'] = th.ones(G_test.num_nodes(), G_test.edata['h'].shape[1])

In [4]:
dgl.save_graphs('test_m.bin', [G_test])

In [5]:
device = th.device('cuda' if th.cuda.is_available() else 'cpu')

In [9]:
device = th.device('cuda' if th.cuda.is_available() else 'cpu')
graphs, _ = dgl.load_graphs('test_m.bin')
G = graphs[0]

In [6]:
G = G.to(device)
node_features = G.ndata['h']
edge_features = G.edata['h']
edge_label = G.edata['label']


In [11]:
print(device)

cuda


In [7]:
print(node_features.shape[0])
print(edge_features.shape[0])

817675
1586064


In [8]:
def compute_accuracy(pred, labels):
    return (pred.argmax(1) == labels).float().mean().item()

class SAGELayer(nn.Module):
    def __init__(self, ndim_in, edims, ndim_out ):
        super(SAGELayer, self).__init__()
        ### force to outut fix dimensions
        self.W_msg = nn.Linear(ndim_in + edims, ndim_out)
        ### apply weight
        self.W_apply = nn.Linear(ndim_in + ndim_out, ndim_out)
        

    def message_func(self, edges):
        return {'m': self.W_msg(th.cat([edges.data['h'], edges.dst['h']], 1))}

    def forward(self, g_dgl, nfeats, efeats):
        with g_dgl.local_scope():
            g = g_dgl
            g.ndata['h'] = nfeats
            g.edata['h'] = efeats
            # Eq4
            g.update_all(self.message_func, fn.mean('m', 'h_neigh'))
            # Eq5          
            g.ndata['h'] = F.relu(self.W_apply(th.cat([g.ndata['h'], g.ndata['h_neigh']], 1)))
            return g.ndata['h']

class MLPPredictor(nn.Module):
    def __init__(self, in_features, edim, out_classes):
        super().__init__()
        self.W = nn.Linear(in_features * 2 + edim, out_classes)

    def apply_edges(self, edges):
        h_u = edges.src['h']
        h_v = edges.dst['h']
        h_e = edges.data['h']
        score = self.W(th.cat([h_u, h_e, h_v], 1))
        return {'score': score}

    def forward(self, graph, h, efeats):
        with graph.local_scope():
            graph.ndata['h'] = h
            graph.edata['h'] = efeats
            graph.apply_edges(self.apply_edges)
            return graph.edata['score']

class Model(nn.Module):
    def __init__(self,  ndim_in, ndim_out, edim):
        super().__init__()
        self.atten = nn.Parameter(th.randn(1, edim))
        self.cov1 = SAGELayer(ndim_in, edim, ndim_out)
        self.cov2 = SAGELayer(ndim_out, edim, ndim_out)
        self.pred = MLPPredictor(ndim_out, edim, 5)
        self.dropout = nn.Dropout(p=0.2)
        
    def forward(self, g, nfeats, efeats):
        efeats = efeats * self.atten
        nfeats = F.relu(self.cov1(g, nfeats, efeats))
        nfeats = self.dropout(nfeats)
        nfeats = F.relu(self.cov2(g, nfeats, efeats))
        nfeats = self.dropout(nfeats)
        return self.pred(g, nfeats, efeats)


In [9]:
from sklearn.utils import class_weight
class_weights = class_weight.compute_class_weight(class_weight='balanced',
                                                 classes=np.unique(G.edata['label'].cpu().numpy()),
                                                 y=G.edata['label'].cpu().numpy())

In [10]:
class_weights = th.FloatTensor(class_weights).cuda()
criterion = nn.CrossEntropyLoss(weight = class_weights)

In [11]:

model = Model( G.ndata['h'].shape[1], 64, G.edata['h'].shape[1]).to(device)
opt = th.optim.Adam(model.parameters())

In [12]:
for epoch in range(500):
    pred = model( G, node_features, edge_features)
    loss = criterion(pred ,edge_label)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (epoch+1) % 100 == 0:
      print('Training acc:', compute_accuracy(pred, edge_label))

Training acc: 0.40481656789779663
Training acc: 0.9740628004074097
Training acc: 0.9790424704551697
Training acc: 0.9784989356994629
Training acc: 0.982431948184967


In [25]:
for epoch in range(200):
    pred = model( G, node_features, edge_features)
    loss = criterion(pred ,edge_label)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (epoch+1) % 100 == 0:
      print('Training acc:', compute_accuracy(pred, edge_label))

Training acc: 0.9820095300674438
Training acc: 0.9792454838752747


In [31]:
for epoch in range(200):
    pred = model( G, node_features, edge_features)
    loss = criterion(pred ,edge_label)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (epoch+1) % 100 == 0:
      print('Training acc:', compute_accuracy(pred, edge_label))

Training acc: 0.981979250907898
Training acc: 0.9821898341178894


In [37]:
for epoch in range(200):
    pred = model( G, node_features, edge_features)
    loss = criterion(pred ,edge_label)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (epoch+1) % 100 == 0:
      print('Training acc:', compute_accuracy(pred, edge_label))

Training acc: 0.9839955568313599
Training acc: 0.9841437339782715


In [44]:
for epoch in range(200):
    pred = model( G, node_features, edge_features)
    loss = criterion(pred ,edge_label)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (epoch+1) % 100 == 0:
      print('Training acc:', compute_accuracy(pred, edge_label))

Training acc: 0.9839810729026794
Training acc: 0.9844048023223877


In [38]:
th.save(model.state_dict(),'A-SAGE-R_m-w-64-1300.pth')

In [19]:
model.load_state_dict(th.load('A-SAGE-R_m.pth'))

<All keys matched successfully>

In [18]:
G_test = G_test.to(device)
node_features_test = G_test.ndata['h']
edge_features_test = G_test.edata['h']
edge_label_test = G_test.edata['label']


In [19]:
print(node_features_test.shape[0])
print(edge_features_test.shape[0])

352108
679744


In [41]:
#w-64-1100
out = model( G_test, node_features_test, edge_features_test)
out = F.softmax(out, dim=1)
_, pred = out.max(dim=1)
print(classification_report(edge_label_test.cpu().numpy(), pred.cpu().numpy(), digits=4))

              precision    recall  f1-score   support

           0     0.8493    0.9926    0.9154      2430
           1     0.9943    0.9943    0.9943    329974
           2     0.9866    0.9795    0.9830    300118
           3     0.9123    0.9437    0.9277     47178
           4     0.2339    0.9091    0.3721        44

    accuracy                         0.9842    679744
   macro avg     0.7953    0.9638    0.8385    679744
weighted avg     0.9846    0.9842    0.9844    679744

